In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import chi2, SelectKBest, RFE, f_classif
from sklearn.metrics import accuracy_score, classification_report
from google.colab import drive

In [4]:
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/datasets/titanic.csv")

y = df["Survived"].astype(int)
x = df.drop(columns=["Survived"])

num_cols = ["Age", "SibSp", "Parch", "Fare"]
cat_cols = ["Pclass", "Sex", "Embarked"]

In [13]:
# feature extraction

num_tf = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaller", StandardScaler())
])

cat_tf = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

In [14]:
# feature construction

x["FamilySize"] = x["SibSp"].fillna(0) + x["Parch"].fillna(0) + 1

preprocess = ColumnTransformer([
    ("num", num_tf, num_cols + ["FamilySize"]),
    ("cat", cat_tf, cat_cols)
])

In [15]:
# feat selection

selection_filter = SelectKBest(score_func=f_classif, k=5)

pipe_filter = Pipeline([
    ("prep", preprocess),
    ("sel", selection_filter),
    ("clf", LogisticRegression(max_iter=1000))
])

In [16]:
# test w/ model

x_train, x_test, y_train, y_test = train_test_split(x,y,stratify=y, test_size=0.2, random_state=42)

pipe_filter.fit(x_train, y_train)
pred = pipe_filter.predict(x_test)
print("=== FILTER (ANOVA) + LR ===")
print("accuracy: ", accuracy_score(y_test,pred))
print(classification_report(y_test,pred))

=== FILTER (ANOVA) + LR ===
accuracy:  0.776536312849162
              precision    recall  f1-score   support

           0       0.80      0.85      0.82       110
           1       0.73      0.67      0.70        69

    accuracy                           0.78       179
   macro avg       0.77      0.76      0.76       179
weighted avg       0.77      0.78      0.77       179



In [18]:
# 1 feat names after preprocess
feat_names = pipe_filter.named_steps["prep"].get_feature_names_out()
print("Feature names:", feat_names)
print("\n")

# 2 mask & score of selected feat (selectkbest)
sel = pipe_filter.named_steps["sel"]
mask = sel.get_support()
selected_names = feat_names[mask]
selected_scores = sel.scores_[mask]
top = sorted(zip(selected_names, selected_scores), key=lambda t: t[1], reverse=True)[:10]
print("Top features:", top)

Feature names: ['num__Age' 'num__SibSp' 'num__Parch' 'num__Fare' 'num__FamilySize'
 'cat__Pclass_1' 'cat__Pclass_2' 'cat__Pclass_3' 'cat__Sex_female'
 'cat__Sex_male' 'cat__Embarked_C' 'cat__Embarked_Q' 'cat__Embarked_S']


Top features: [('cat__Sex_female', np.float64(306.5932488951883)), ('cat__Sex_male', np.float64(306.59324889518797)), ('cat__Pclass_3', np.float64(80.33862734392042)), ('cat__Pclass_1', np.float64(73.99727564291717)), ('num__Fare', np.float64(58.31490728198491))]
